# Multivariate Time-to-Event Transformer

**A from-scratch PyTorch tutorial.** You will build a causal multivariate
transformer with a Weibull time-to-event head and apply it to NASA's
CMAPSS jet engine degradation dataset.

## Learning objectives

By the end of this notebook you will be able to:

1. Explain why a **causal transformer** is the right shape for a streaming
   sensor problem
2. Build, from scratch, every component of a transformer: continuous-input
   embedding, sinusoidal positional encoding, scaled dot-product attention
   with masking, multi-head attention, pre-norm transformer blocks
3. Parameterize a **probability distribution over time-to-event** using a
   Weibull output head
4. Train the model with a **negative log-likelihood loss that handles
   right-censored data** (units that did not run to failure)
5. Convert distribution predictions into **calibrated point estimates with
   confidence intervals** — i.e. "engine N will fail in X cycles (90% CI:
   low–high)"
6. Evaluate the model with proper survival metrics (concordance index,
   Brier score, calibration plots)
7. Extend the architecture with **additional output heads** without
   rebuilding the encoder

## How to work through this notebook

- Markdown cells (like this one) are fully written — they teach the concept
- Code cells contain **function signatures, docstrings, and TODO hints**.
  You write the implementation.
- Cells are ordered so each one depends only on those above it
- Math is presented intuition-first; rigorous derivations are in
  clearly-marked sections you can skip on a first pass

Let's begin.


## Top-level architecture

```mermaid
flowchart TB
    A["Raw sensors x_t in R^F<br/>3 ops settings + 21 sensors"] --> B["Regime-aware normalization<br/>(cluster ops settings → per-cluster z-score)"]
    B --> C["Continuous embedding<br/>Linear: F → d_model"]
    C --> D["+ Sinusoidal positional encoding"]
    D --> E["N × Causal Transformer Block<br/>(masked MHA → FFN, pre-norm)"]
    E --> F["Per-timestep hidden states h_t in R^d_model"]
    F --> G1["Weibull head<br/>outputs (k_t, λ_t)"]
    F --> G2["YourHeadHere stub<br/>(extension point)"]
    G1 --> L1["Weibull NLL<br/>with right-censoring"]
    G2 --> L2["(your loss)"]
    L1 --> SUM["Weighted total loss"]
    L2 --> SUM
    G1 --> Q["Quantile conversion<br/>median + 90% CI"]
    Q --> R["Per-engine forecast:<br/>'fails in X cycles (low–high)'"]
```

Every box in this diagram corresponds to a code cell below. The
**Weibull head** plus **NLL with censoring** is the heart of the
"time-to-event" framing — instead of predicting a single number, we
predict a full probability distribution over when the system will fail.


## Why a *causal* transformer?

A causal (autoregressive) transformer constrains attention so each
timestep can only look at past and present sensor readings. This matters
for three reasons:

**1. It prevents leakage.** "Time to event" means predicting the future.
If attention could peek at future degradation patterns, the model would
"cheat" during training and fall apart at inference time.

**2. It mirrors the inference setting.** In production, sensors stream in
one cycle at a time. A causal model gives a fresh forecast every cycle
without needing to re-process the full history.

**3. It gives dense supervision.** A non-causal model gets *one* training
target per engine (the final failure cycle). A causal model gets *one
target per timestep* — a 200-cycle engine becomes 200 supervised samples.
Same data, two orders of magnitude more gradient signal.

```mermaid
flowchart LR
    subgraph "Bidirectional (one target per engine)"
        B1[t=1] --> B2[t=2] --> B3[...] --> Bn[t=N]
        Bn --> ByT[predict T=N]
    end
    subgraph "Causal (one target per timestep)"
        C1[t=1] --> Cy1[predict T-1]
        C2[t=2] --> Cy2[predict T-2]
        Cn[t=N] --> CyN[predict T-N = 0]
    end
```


## Why a *Weibull* head?

The phrase "time to event" comes from survival analysis. The right output
isn't a single number — it's a **probability distribution over event
times**. The Weibull distribution is a natural choice because:

- It only takes positive values (event times can't be negative)
- It has only two parameters: a **shape** $k$ and a **scale** $\lambda$
- It's flexible: $k<1$ gives decreasing-hazard ("survivor bias"), $k=1$
  gives a memoryless exponential, $k>1$ gives the increasing-hazard
  ("wear out") behavior we expect from jet engines
- The PDF, CDF, and survival function all have closed forms — gradients
  flow cleanly through the negative log-likelihood

The head outputs $(k_t, \lambda_t)$ at every timestep, parameterizing
the distribution of remaining cycles **given everything observed up to
time $t$**.

<details>
<summary><b>Click here for the rigorous math (you can skip this on first pass)</b></summary>

The Weibull probability density, cumulative distribution, and survival
functions are:

$$f(t; k, \lambda) = \frac{k}{\lambda}\left(\frac{t}{\lambda}\right)^{k-1}
                     \exp\!\left(-\left(\frac{t}{\lambda}\right)^{k}\right),
                     \qquad t \ge 0$$

$$F(t; k, \lambda) = 1 - \exp\!\left(-\left(\frac{t}{\lambda}\right)^{k}\right)$$

$$S(t; k, \lambda) = 1 - F(t; k, \lambda) = \exp\!\left(-\left(\frac{t}{\lambda}\right)^{k}\right)$$

The negative log-likelihood for a single observation $(t_i, u_i)$, where
$u_i \in \{0, 1\}$ indicates whether the event was observed ($u_i = 1$)
or right-censored ($u_i = 0$), is:

$$\mathcal{L}_i = -\,u_i \log f(t_i; k_i, \lambda_i)
                  -\,(1 - u_i) \log S(t_i; k_i, \lambda_i)$$

Substituting and simplifying gives a form that is numerically stable on
the log scale (which is what we'll implement in Part 4).

</details>

The Weibull head and its loss live in Parts 3 and 4. Part 5 converts the
$(k, \lambda)$ predictions back into the answer the human wanted in the
first place: "this engine will fail in **X** cycles, give or take."


## Roadmap

| Part | Topic | What you'll do |
|---|---|---|
| **0** | Architecture overview | (this part — already done) |
| **1** | Setup | Install dependencies, set seeds, pick a device |
| **2** | Data | Download CMAPSS, normalize per operating regime, build padded tensor batches with causal targets |
| **3** | Model components | Embed continuous sensors → positional encoding → scaled dot-product attention → multi-head → transformer block → full model with head registry |
| **4** | Training | Weibull NLL with censoring → AdamW + cosine schedule → training loop → loss curves |
| **5** | Evaluation | C-index, Brier score, **distribution → number with error bars**, calibration plots |
| **6** | Extensions | Adding more heads, more datasets, attention visualization |

Onward.


---
# Part 1 — Setup

We need PyTorch, a handful of supporting libraries, reproducible random
seeds, and a compute device. Nothing exciting yet — but a clean setup
saves hours of debugging later.

## Why these dependencies

- **`torch`** — the deep learning framework we're building on
- **`numpy`**, **`pandas`** — array and tabular data handling
- **`scikit-learn`** — KMeans for operating-regime clustering (Part 2)
- **`scipy`** — gamma function for Weibull mean (Part 5)
- **`matplotlib`** — plots
- **`tqdm`** — progress bars
- **`kaggle`** — auto-download of the CMAPSS dataset

(The survival metrics in Part 5 — concordance index and Brier score —
are implemented from scratch with numpy, consistent with the rest of
the notebook.)

## Device selection

This notebook runs on CPU, CUDA (NVIDIA GPU), or MPS (Apple Silicon).
The setup cell below picks the best available automatically. The model
is small enough that CPU is fine for a first training run.


In [ ]:
# Run once. Skip if you've already installed the requirements.
# (Re-running is harmless — pip will report 'already satisfied'.)
%pip install -q -r requirements.txt


In [ ]:
"""Imports, seed setting, and device detection.

Implementation hints:
    1. import torch, numpy, pandas, random
       from matplotlib import pyplot
       (nn and functional are imported where they're used in Part 3)
    2. Set seeds for python's random, numpy, and torch (CPU + CUDA)
    3. Detect device:
         - torch.cuda.is_available() → "cuda"
         - torch.backends.mps.is_available() → "mps"
         - else → "cpu"
    4. Print torch version and chosen device so you can sanity-check
"""

# imports
import torch, numpy, pandas, random
from matplotlib import pyplot

# Seed setting (a helper function `set_seed(seed: int)` is nice)
def set_seed(seed: int):
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Device detection — store as a module-level `DEVICE` constant
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# Print torch version and DEVICE
print(f"Torch version: {torch.__version__}")
print(f"Using device: {DEVICE}")


---
# Part 2 — Data

## The CMAPSS dataset

CMAPSS (Commercial Modular Aero-Propulsion System Simulation) is a
simulated turbofan engine dataset published by NASA's Prognostics Center
of Excellence. It contains **four subsets** of varying difficulty; we'll
focus on **FD002**.

| Subset | Train units | Test units | Operating conditions | Fault modes |
|---|---|---|---|---|
| FD001 | 100 | 100 | 1 | 1 (HPC degradation) |
| **FD002** | **260** | **259** | **6** | **1 (HPC degradation)** |
| FD003 | 100 | 100 | 1 | 2 (HPC + fan degradation) |
| FD004 | 248 | 249 | 6 | 2 |

Each row in `train_FD002.txt` is one operating cycle of one engine and
has 26 columns:

| Columns | Meaning |
|---|---|
| 1 | `unit` — engine ID |
| 2 | `cycle` — operating cycle (1, 2, 3, …) |
| 3–5 | `op_setting_1..3` — operational conditions (altitude, Mach, throttle resolver angle) |
| 6–26 | `sensor_1..21` — sensor measurements |

**Crucial difference between train and test:**

- In **`train_FD002.txt`**, each engine runs **until failure**. The last
  cycle of each engine *is* the failure event. We say these observations
  are **uncensored**.
- In **`test_FD002.txt`**, each engine is stopped **before** failure. We
  only know the engine survived at least the observed number of cycles.
  The true remaining cycles after the cutoff are given in
  `RUL_FD002.txt`. These observations are **right-censored**.

That asymmetry is the whole reason we need a censoring-aware loss.


In [ ]:
"""Auto-download CMAPSS from Kaggle, with a documented NASA fallback.

Implementation hints:
    1. Check whether ./data/CMAPSSData/train_FD002.txt already exists.
       If yes, print "already present" and return.
    2. Otherwise, use kaggle CLI to download `behrad3d/nasa-cmaps`:
           kaggle datasets download -d behrad3d/nasa-cmaps -p ./data --unzip
       (Requires ~/.kaggle/kaggle.json configured — see data/README.md.)
    3. If kaggle fails (no creds, no internet, etc.), print the NASA
       manual fallback URL from data/README.md and raise a helpful error.

Notes:
    - The Kaggle dataset extracts into `data/CMAPSSData/` directly.
    - Use subprocess.run with check=False so you can detect failure
      and print the fallback message.
"""

import subprocess
from pathlib import Path

DATA_DIR = Path("data") / "CMAPSSData"


def download_cmapss() -> None:
    # TODO: short-circuit if files already exist
    # TODO: shell out to kaggle CLI
    # TODO: on failure, print fallback URL and raise
    raise NotImplementedError("Implement CMAPSS auto-download")


# TODO: call download_cmapss()


In [ ]:
"""Load FD002 train, test, and RUL files into pandas DataFrames.

Implementation hints:
    1. Define column names once:
           ["unit", "cycle"]
         + [f"op_setting_{i}" for i in range(1, 4)]
         + [f"sensor_{i}" for i in range(1, 22)]
       → 26 columns total
    2. Use pandas.read_csv with sep=r"\s+", header=None, names=COLS.
    3. RUL file is one integer per line — also pandas.read_csv but with
       a single "rul" column.

Return three DataFrames: train_df, test_df, rul_df.
"""

SUBSET = "FD002"
COLS = (
    ["unit", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)


def load_cmapss(subset: str = SUBSET):
    # TODO: read train, test, rul into DataFrames
    raise NotImplementedError("Load CMAPSS into DataFrames")


# TODO: call load_cmapss and store as train_df, test_df, rul_df
# TODO: print shapes as a sanity check


## Pipeline overview

```mermaid
flowchart LR
    A["train_FD002.txt<br/>test_FD002.txt<br/>RUL_FD002.txt"] --> B["pandas DataFrames"]
    B --> C["EDA<br/>(plot sensors, lengths,<br/>ops scatter)"]
    B --> D["Regime clustering<br/>(KMeans on ops)"]
    D --> E["Per-regime<br/>z-score normalization"]
    E --> F["Sensor selection<br/>(drop near-constant)"]
    F --> G["Causal target construction<br/>(t, target_cycles_remaining,<br/>censored_flag)"]
    G --> H["PyTorch Dataset"]
    H --> I["DataLoader with<br/>padding + attention mask"]
```

Each box below corresponds to a markdown + code cell pair. You'll move
from "raw text files" to "batched padded tensors with masks" in about
ten cells.


## Exploratory data analysis

Before building features, take a look at the data:

- **Sequence length histogram** — how long do engines run? Is there a
  big spread?
- **Sample sensor traces** — pick a few engines, plot a few sensors
  across their lifetime. Do you see clear degradation patterns?
- **Operating conditions scatter** — plot the three `op_setting_*`
  columns. Do they cluster into discrete regimes?

EDA isn't optional — the regime clustering in the next cell depends on
your having visual confirmation that distinct operating regimes exist.


In [ ]:
"""Three EDA plots: sequence length histogram, sample sensor traces,
operating-condition scatter.

Implementation hints:
    1. Sequence length per engine:
           lengths = train_df.groupby("unit")["cycle"].max()
           pyplot.hist(lengths, bins=30)
    2. Sample sensor traces: pick 3-5 engines, plot sensors 2, 7, 11
       (these tend to show clear degradation in FD002) over cycles.
    3. Ops scatter: scatter op_setting_1 vs op_setting_2, color by
       op_setting_3 (or use 3D). You should see ~6 clusters.
"""

# TODO: sequence length histogram
# TODO: sensor traces for a few engines
# TODO: ops scatter plot
raise NotImplementedError("Make the three EDA plots")


## Why FD002 needs regime-aware normalization

FD002 contains **six distinct operating conditions** (combinations of
altitude, Mach number, and throttle resolver angle). The sensor readings
differ wildly across regimes — a "high reading" at cruise altitude might
be a "low reading" at sea level.

Z-scoring the sensors **globally** would mix regimes together and
destroy the degradation signal:

```mermaid
flowchart LR
    A["Raw sensors<br/>(mixed regimes)"] --> B{"Normalize<br/>globally?"}
    B -- "Yes (wrong)" --> C["Z-score destroys<br/>regime structure"]
    B -- "No: per-regime" --> D["KMeans on<br/>op_setting_1..3"]
    D --> E["Assign each row<br/>a regime ID"]
    E --> F["Z-score each sensor<br/>WITHIN its regime"]
    F --> G["Now all values<br/>are in 'regime-relative' units"]
```

The fix: cluster the ops settings into 6 groups with KMeans, then
z-score each sensor *within* its regime. After this, a positive value
always means "above-normal for this regime" — comparable across regimes.


In [ ]:
"""Cluster operating settings into 6 regimes and z-score each sensor
within its regime.

Implementation hints:
    1. KMeans(n_clusters=6, random_state=42, n_init=10) on the
       op_setting_{1,2,3} columns from the TRAIN df.
    2. Use the FITTED KMeans to assign regime IDs to BOTH train and test
       (do not refit on test).
    3. For each (regime, sensor) pair, compute mean and std on the train
       set only. Store as a dict {regime: {sensor: (mean, std)}}.
    4. Apply these statistics to BOTH train and test (do not recompute
       statistics on test — that would leak).
    5. Return train_df_norm, test_df_norm with new columns sensor_1..21
       overwritten with z-scored values and a new column 'regime'.

Why fit-on-train, apply-to-test: any preprocessing parameters learned
from the test set leak future information into your evaluation.
"""

from sklearn.cluster import KMeans

SENSOR_COLS = [f"sensor_{i}" for i in range(1, 22)]
OP_COLS = [f"op_setting_{i}" for i in range(1, 4)]


def fit_regime_normalizer(train_df, n_regimes: int = 6):
    """Fit KMeans on ops settings and per-regime sensor statistics."""
    # TODO: fit KMeans on op settings
    # TODO: assign regime to train_df
    # TODO: compute per-(regime, sensor) mean and std on train
    # TODO: return (kmeans, stats_dict)
    raise NotImplementedError("Fit the regime normalizer")


def apply_regime_normalizer(df, kmeans, stats):
    """Assign regimes using fitted KMeans, z-score sensors per regime."""
    # TODO: assign regimes
    # TODO: z-score each sensor per regime using stats
    # TODO: return the modified copy
    raise NotImplementedError("Apply the regime normalizer")


# TODO: fit on train, apply to both train and test


## Sensor selection

CMAPSS has 21 sensors but **several of them are near-constant** within a
regime — they carry no degradation signal. Including them just adds
parameters and noise.

We'll keep any sensor whose post-normalization standard deviation is
greater than a small threshold (e.g., 0.05). The dropped sensors are
reported so the decision is visible, not silent.


In [ ]:
"""Drop near-constant sensors after normalization.

Implementation hints:
    1. Compute the std of each post-normalization sensor on the train set.
    2. Keep sensors where std > threshold (try 0.05).
    3. Report which sensors were dropped and which were kept.
    4. Apply the same selection to both train and test.

Return: list of kept sensor column names.
"""


def select_informative_sensors(train_df, threshold: float = 0.05):
    # TODO: compute std per sensor
    # TODO: pick sensors above threshold
    # TODO: print kept and dropped
    raise NotImplementedError("Select informative sensors")


# TODO: call and store as KEPT_SENSORS (list of column names)


## Causal target construction

This is the conceptual heart of the data pipeline. The model is causal,
so we want **one supervised sample per timestep, per engine**.

For a training engine that ran for $N$ cycles total, at each timestep
$t \in \{1, \dots, N\}$ the target time-to-event is $N - t$ — that many
cycles remain until failure. Because the engine ran to failure, every
sample is **uncensored** ($u = 1$).

For test engines, the engine was stopped after $T$ observed cycles, and
the true remaining cycles is given in `RUL_FD002.txt`. At the final
observed timestep $T$, all we can say is that the engine survived at
least the recorded RUL — that's a **right-censored** sample ($u = 0$).

```mermaid
flowchart TB
    subgraph "Train engine (uncensored)"
        T1["Engine ran 200 cycles, then failed"]
        T1 --> T2["At t=1: target = 199, u = 1"]
        T1 --> T3["At t=50: target = 150, u = 1"]
        T1 --> T4["At t=200: target = 0, u = 1"]
    end
    subgraph "Test engine (right-censored)"
        E1["Engine observed for 130 cycles, true RUL = 80"]
        E1 --> E2["At t=130 (final): target = 80, u = 0"]
    end
```

For simplicity, during training we use the full causal target stream per
engine; during evaluation we score on the final-cycle prediction per
test engine (matching the standard CMAPSS evaluation protocol).


In [ ]:
"""Build (sequence, target_time, censored_flag) tuples.

The training set is a list of (sensor_sequence, targets, censored_flags)
where:
    - sensor_sequence: numpy.ndarray of shape (T_i, F) for engine i
    - targets: numpy.ndarray of shape (T_i,) with cycles-remaining at each t
    - censored_flags: numpy.ndarray of shape (T_i,) — for train, all zeros
      meaning UNCENSORED everywhere (event observed at end)

NOTE on the convention: we'll use u=1 for OBSERVED and u=0 for CENSORED
in the loss (Part 4). Some libraries reverse this — be consistent with
your own choice and document it.

Implementation hints:
    1. For each engine in train_df, sort by cycle, take only KEPT_SENSORS
       columns, build the (T, F) array and the corresponding target array
       N-cycle (where N is the engine's failure cycle).
    2. censored_flags = numpy.zeros(T, dtype=numpy.int64)   # all observed
    3. For test, do the same but truncate at the test cutoff; final-step
       target = rul_df[engine_id]; censored_flags = numpy.ones at the final
       step only, zeros elsewhere (the within-trajectory targets are
       imputed cycles-from-now, optionally treated as observed too).

       NOTE: For a strict survival framing, you could mark every test
       observation as censored (we only ever see them survive). For
       this learning notebook we keep within-trajectory cycles uncensored
       (cycles-remaining within the observed window IS known) and treat
       the LAST cycle as censored at the RUL value.
"""

import numpy


def build_causal_tuples(df, kept_sensor_cols, is_train: bool,
                        rul_df=None):
    samples = []
    for unit, eng in df.groupby("unit"):
        eng = eng.sort_values("cycle")
        # TODO: extract sensor_sequence as a (T, F) numpy.ndarray
        # TODO: build targets array
        # TODO: build censored_flags array
        # TODO: append (sensor_sequence, targets, censored_flags) to samples
        raise NotImplementedError("Build a causal tuple for one engine")
    return samples


# TODO: build train_samples and test_samples


## PyTorch Dataset + DataLoader with padding

Engines have very different lengths (FD002 ranges from roughly 130 to
nearly 380 cycles). We need to batch them together, which means:

1. **Pad** each sequence in a batch up to the batch's longest length
2. Provide a **padding mask** so attention ignores padded positions
3. Combine the padding mask with the **causal mask** when computing
   attention (Part 3)

A standard pattern: a `Dataset` that returns one engine at a time, and a
custom `collate_fn` that pads + builds the mask at batch time.


In [ ]:
"""PyTorch Dataset and collate_fn for variable-length causal training.

Implementation hints (Dataset):
    1. __init__(self, samples) — store the list
    2. __len__ returns number of engines
    3. __getitem__(idx) returns a dict with keys:
         "sensors":  torch.tensor of shape (T, F), float32
         "targets":  torch.tensor of shape (T,), float32
         "censored": torch.tensor of shape (T,), float32 (0 or 1)

Implementation hints (collate_fn):
    1. Find T_max in the batch
    2. Pre-allocate padded tensors of shape (B, T_max, F) etc.
       (fill with zeros for sensors and targets, fill censored with 0)
    3. Build a key_padding_mask of shape (B, T_max), True where PADDED
       (i.e., for positions BEYOND the original length)
    4. Return a dict with: sensors, targets, censored, key_padding_mask,
       lengths
"""

import torch
from torch.utils.data import Dataset


class CMAPSSDataset(Dataset):
    def __init__(self, samples):
        # TODO
        raise NotImplementedError

    def __len__(self):
        # TODO
        raise NotImplementedError

    def __getitem__(self, idx):
        # TODO: return a dict of tensors
        raise NotImplementedError


def collate_fn(batch):
    # TODO: find T_max
    # TODO: allocate padded tensors
    # TODO: fill from each item
    # TODO: build key_padding_mask (True where padded)
    # TODO: return the batch dict
    raise NotImplementedError("Implement collate_fn with padding mask")


In [ ]:
"""Wire up the DataLoaders.

Implementation hints:
    - Use a moderate batch size (e.g., 32 or 64).
    - shuffle=True for train, shuffle=False for test.
    - num_workers=0 in a notebook is usually safest.
    - Pass collate_fn explicitly.

Then pull a single batch and print the shape of every tensor in it as a
sanity check before moving on to the model.
"""

from torch.utils.data import DataLoader

BATCH_SIZE = 32

# TODO: train_dataset, test_dataset
# TODO: train_loader, test_loader
# TODO: pull a sample batch and print shapes
raise NotImplementedError("Build DataLoaders and print sample batch shapes")


---
# Part 3 — Model components

Now the fun part. We'll build every component of the transformer from
scratch, in dependency order:

```mermaid
flowchart TB
    A[Continuous embedding<br/>3.2] --> B[Positional encoding<br/>3.4]
    B --> C[Scaled dot-product attention<br/>3.6]
    C --> D[Multi-head attention<br/>3.8]
    D --> E[Feed-forward<br/>3.10]
    E --> F[Transformer block<br/>3.12]
    F --> G[Head registry<br/>3.14, 3.15]
    G --> H[TTETransformer<br/>3.16]
```

Each component gets its own intuition, math (where appropriate), and a
scaffolded code cell.


## 3.1  Continuous-input embedding

NLP transformers embed discrete tokens (words) by looking them up in a
learned table. Sensor data has no such vocabulary — each input is a
real-valued vector $x_t \in \mathbb{R}^F$ where $F$ is the number of
selected sensors.

The simplest (and surprisingly effective) approach is a **linear
projection**: $e_t = W x_t + b$ with $W \in \mathbb{R}^{d_{\text{model}} \times F}$.

This learns to mix sensors into the latent space the attention layers
will operate in. Note that $F$ here is small (we kept only informative
sensors in Part 2) so $W$ has just a few hundred parameters.


In [ ]:
"""Linear projection from F sensors to d_model latent dimension.

Implementation hints:
    1. Subclass nn.Module
    2. Hold an nn.Linear(F, d_model)
    3. forward(x): x has shape (B, T, F), return (B, T, d_model)
"""

from torch import nn


class ContinuousEmbedding(nn.Module):
    def __init__(self, num_features: int, d_model: int):
        super().__init__()
        # TODO: nn.Linear(num_features, d_model)
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError


## 3.3  Sinusoidal positional encoding

Attention is **permutation-invariant** — without position information, it
treats sensor readings as an unordered set. Positional encoding adds a
position-dependent vector to each embedding so the model knows "this
came before that."

The Vaswani-style sinusoidal encoding uses sines and cosines at
geometrically-spaced frequencies:

$$\text{PE}_{(t, 2i)}   = \sin(t \cdot 10000^{-2i/d_{\text{model}}})$$
$$\text{PE}_{(t, 2i+1)} = \cos(t \cdot 10000^{-2i/d_{\text{model}}})$$

Two nice properties:

- Each dimension has a sinusoidal frequency from short (high frequency)
  to long (low frequency) — like a positional Fourier basis
- The encoding for position $t + k$ can be expressed as a linear
  function of the encoding for position $t$ — making relative positions
  linearly accessible

We precompute the full encoding table once and add it to the embedding.


In [ ]:
"""Sinusoidal positional encoding (Vaswani et al., 2017).

Implementation hints:
    1. In __init__, precompute a (max_len, d_model) tensor pe.
    2. positions = torch.arange(max_len).unsqueeze(1)
       div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
       pe[:, 0::2] = sin(positions * div_term)
       pe[:, 1::2] = cos(positions * div_term)
    3. Register pe as a buffer (not a parameter) so it moves with .to(device)
       but doesn't get trained.
    4. forward(x): x has shape (B, T, d_model); return x + pe[:T] (broadcast).
"""

import math


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        # TODO: build pe of shape (max_len, d_model)
        # TODO: self.register_buffer("pe", pe)
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: add positional encoding for the first T positions
        raise NotImplementedError


## 3.5  Scaled dot-product attention

Attention answers "for each position $t$, what mixture of *other*
positions' values do I need?" Given queries $Q$, keys $K$, and values
$V$:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}} + M\right) V$$

```mermaid
flowchart LR
    Q[Q  B,H,T,d_k] --> S["scores =<br/>Q @ K^T / √d_k"]
    K[K  B,H,T,d_k] --> S
    S --> M["+ mask<br/>(set masked positions to -∞)"]
    M --> SM["softmax over keys axis"]
    SM --> O["output =<br/>attn @ V"]
    V[V  B,H,T,d_v] --> O
```

The scaling by $\sqrt{d_k}$ keeps the softmax sharp regardless of $d_k$ —
without it, gradients vanish for large $d_k$.

**Masking** is how we enforce causality: we set scores at positions the
query "shouldn't see" to $-\infty$ before the softmax. There are TWO
masks we'll combine:

- **Causal mask**: query at position $t$ can't attend to key positions
  $> t$ (upper-triangular)
- **Padding mask**: queries should not attend to padded positions in
  shorter sequences within a batch


In [ ]:
"""Scaled dot-product attention with combined causal + padding mask.

Args:
    q: (B, H, T, d_k)
    k: (B, H, T, d_k)
    v: (B, H, T, d_v)
    mask: bool tensor of shape broadcastable to (B, H, T, T).
          True = MASKED (do not attend). False = attend.
          May be None.

Returns:
    output: (B, H, T, d_v)
    attn:   (B, H, T, T)  — useful for visualization (Part 6 exercise)

Implementation hints:
    1. d_k = q.size(-1)
    2. scores = q @ k.transpose(-2, -1) / sqrt(d_k)
    3. if mask is not None:  scores = scores.masked_fill(mask, float("-inf"))
    4. attn = functional.softmax(scores, dim=-1)
    5. output = attn @ v
"""

from torch.nn import functional


def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    # TODO: scaled scores
    # TODO: apply mask
    # TODO: softmax + matmul with v
    raise NotImplementedError("Implement scaled_dot_product_attention")


## 3.7  Multi-head attention

A single attention head can only attend to ONE thing per position. Real
sequences have multiple useful "things" to attend to (e.g., short-term
trends vs. long-term degradation). Multi-head attention runs several
attention computations in parallel in lower-dimensional subspaces.

```mermaid
flowchart TB
    X["x: (B, T, d_model)"] --> WQ["W_Q"]
    X --> WK["W_K"]
    X --> WV["W_V"]
    WQ --> Q["Q: (B, T, d_model)"]
    WK --> K["K: (B, T, d_model)"]
    WV --> V["V: (B, T, d_model)"]
    Q --> R["reshape → (B, H, T, d_k)<br/>where d_k = d_model / H"]
    K --> R
    V --> R
    R --> A["scaled_dot_product_attention<br/>across all H heads in parallel"]
    A --> C["concat heads → (B, T, d_model)"]
    C --> WO["W_O: (B, T, d_model)"]
```

The reshape from `(B, T, d_model)` to `(B, H, T, d_k)` is the trick:
because attention treats the leading batch and head dimensions
identically, we get H parallel attentions in a single matrix multiply.


In [ ]:
"""Multi-head attention.

Args (forward):
    x: (B, T, d_model)
    key_padding_mask: (B, T) bool, True where padded
    causal: bool — whether to apply causal masking

Returns:
    output: (B, T, d_model)
    attn:   (B, H, T, T)

Implementation hints (forward):
    1. Project x through three linear layers to get Q, K, V of shape
       (B, T, d_model).
    2. Reshape each to (B, T, H, d_k) then transpose to (B, H, T, d_k).
    3. Build the mask:
         - causal_mask = (B=1, H=1, T, T) upper-triangular bool with
           True ABOVE the diagonal
         - pad_mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
           shape (B, 1, 1, T) — broadcasts to (B, H, T, T)
         - combine with logical OR
    4. Call scaled_dot_product_attention.
    5. Concatenate heads: transpose to (B, T, H, d_k) → reshape to
       (B, T, d_model).
    6. Project back through W_O.
"""


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        # TODO: assert d_model % num_heads == 0
        # TODO: self.d_k = d_model // num_heads
        # TODO: three nn.Linear for Q, K, V; one for output W_O
        raise NotImplementedError

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: torch.Tensor | None = None,
        causal: bool = True,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO
        raise NotImplementedError


## 3.9  Position-wise feed-forward

After attention mixes information across positions, each position gets
its own little 2-layer MLP applied independently:

$$\text{FFN}(x) = W_2 \,\sigma(W_1 x + b_1) + b_2$$

The hidden dimension $d_{\text{ff}}$ is typically $4 \cdot d_{\text{model}}$.
The non-linearity $\sigma$ is GELU (modern default) — it's smoother
than ReLU and tends to train better.

This is where most of the model's parameters live, but the operation is
position-wise (no mixing across timesteps), making it very fast on GPU.


In [ ]:
"""Position-wise feed-forward network.

Implementation hints:
    - Linear(d_model, d_ff) → GELU → Dropout → Linear(d_ff, d_model)
    - GELU: nn.GELU() (or functional.gelu)
"""


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError


## 3.11  Transformer block (pre-norm)

The "transformer block" stacks attention and feed-forward with residual
connections and layer normalization. There are two layouts:

- **Post-norm** (Vaswani 2017): $x \leftarrow \text{LN}(x + \text{sublayer}(x))$
- **Pre-norm** (modern default): $x \leftarrow x + \text{sublayer}(\text{LN}(x))$

We use pre-norm because it trains more stably at depth and is what every
modern LLM uses.

```mermaid
flowchart TB
    X[x] --> N1[LayerNorm]
    N1 --> MHA[Masked MHA]
    MHA --> D1[Dropout]
    D1 --> A1((+))
    X --> A1
    A1 --> N2[LayerNorm]
    N2 --> FFN[FeedForward]
    FFN --> D2[Dropout]
    D2 --> A2((+))
    A1 --> A2
    A2 --> Y[output]
```

This is the same structure repeated $N$ times in the encoder.


In [ ]:
"""One pre-norm transformer block.

Implementation hints:
    Forward path (pre-norm):
        a = self.mha(self.ln1(x), key_padding_mask=..., causal=True)[0]
        x = x + self.dropout(a)
        f = self.ffn(self.ln2(x))
        x = x + self.dropout(f)
        return x
"""


class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        # TODO: self.ln1, self.mha, self.ln2, self.ffn, self.dropout
        raise NotImplementedError

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        # TODO
        raise NotImplementedError


## 3.13  The head registry pattern

To support **multiple output heads from a shared encoder**, we use a
simple registry pattern: each head is an `nn.ModuleDict` entry with its
own forward and an associated loss.

```mermaid
flowchart LR
    H["per-timestep hidden states<br/>(B, T, d_model)"] --> R{"Head Registry"}
    R --> W["WeibullHead<br/>→ (k, λ)<br/>→ weibull_nll_loss"]
    R --> Y["YourHeadHere<br/>(stub)"]
    W --> L1["loss_weibull"]
    Y --> L2["loss_yours"]
    L1 --> S["Σ w_i · loss_i"]
    L2 --> S
```

Adding a new head means:

1. Define a new `nn.Module` that takes `(B, T, d_model)` and returns a
   prediction
2. Add it to the registry by name
3. Define a per-head loss function
4. Give it a weight in the total-loss aggregation

The encoder doesn't change. This is exactly the multi-head
extensibility the project requirements call for.


In [ ]:
"""WeibullHead: outputs (k, λ) per timestep with strict positivity.

Implementation hints:
    1. Two nn.Linear(d_model, 1) layers: one for the k pre-activation,
       one for the lambda pre-activation.
    2. Apply softplus to make both strictly positive:
           k = functional.softplus(self.k_proj(h)) + 1e-6
           lam = functional.softplus(self.lam_proj(h)) + 1e-6
       The 1e-6 floor prevents log(0) in the loss.
    3. Returns a dict: {"k": (B, T), "lambda": (B, T)}.

Why softplus over exp:
    - softplus is more numerically stable for large activations.
    - exp can produce huge values that blow up the gradient.
"""


class WeibullHead(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        # TODO: two linear layers (d_model -> 1) each
        raise NotImplementedError

    def forward(self, h: torch.Tensor) -> dict[str, torch.Tensor]:
        # TODO: project, softplus, squeeze last dim, return dict
        raise NotImplementedError


In [ ]:
"""YourHeadHere: documented extension point.

This is a stub showing how to add another output head. Possible heads
you might add later:
    - Operating-regime classifier (predict which of 6 regimes)
    - Sensor-reconstruction head (predict next-step sensor vector)
    - Health-state classifier (healthy / degrading / critical)
    - Anomaly score (single positive scalar per timestep)

To add a real head:
    1. Replace the body of forward() with your head's computation.
    2. Define a matching loss function in Part 4.
    3. Register it under a unique name in TTETransformer.heads.
    4. Give it a weight in compute_total_loss.
"""


class YourHeadHere(nn.Module):
    def __init__(self, d_model: int, output_dim: int = 1):
        super().__init__()
        # TODO: define the layers your head needs
        # Example for a regression head:
        #     self.proj = nn.Linear(d_model, output_dim)
        raise NotImplementedError("Stub — replace with your head")

    def forward(self, h: torch.Tensor) -> dict[str, torch.Tensor]:
        """h: (B, T, d_model) → dict of per-timestep predictions."""
        # TODO: compute your prediction and return it in a dict
        raise NotImplementedError("Stub — replace with your head")


In [ ]:
"""TTETransformer: the full model.

Wires together:
    embedding → positional encoding → N transformer blocks
    → head registry (dict of named heads)

Returns a dict {head_name: head_outputs}.

Implementation hints (forward):
    1. h = self.embedding(sensors)
    2. h = self.pos_enc(h)
    3. for block in self.blocks: h = block(h, key_padding_mask=...)
    4. outputs = {name: head(h) for name, head in self.heads.items()}
    5. return outputs

Suggested defaults:
    num_features = len(KEPT_SENSORS)
    d_model = 64
    num_heads = 4
    d_ff = 256
    num_layers = 3
    dropout = 0.1
"""


class TTETransformer(nn.Module):
    def __init__(
        self,
        num_features: int,
        d_model: int = 64,
        num_heads: int = 4,
        d_ff: int = 256,
        num_layers: int = 3,
        dropout: float = 0.1,
        max_len: int = 1000,
        extra_heads: dict[str, nn.Module] | None = None,
    ):
        super().__init__()
        # TODO: self.embedding = ContinuousEmbedding(...)
        # TODO: self.pos_enc = PositionalEncoding(...)
        # TODO: self.blocks = nn.ModuleList([TransformerBlock(...) for _ in range(num_layers)])
        # TODO: build head registry
        #     heads = {"weibull": WeibullHead(d_model)}
        #     if extra_heads: heads.update(extra_heads)
        #     self.heads = nn.ModuleDict(heads)
        raise NotImplementedError

    def forward(
        self,
        sensors: torch.Tensor,           # (B, T, F)
        key_padding_mask: torch.Tensor | None = None,  # (B, T) bool, True where padded
    ) -> dict[str, dict[str, torch.Tensor]]:
        # TODO
        raise NotImplementedError


In [ ]:
"""Sanity check: forward pass on a synthetic batch.

Build a TTETransformer, feed it a random tensor of shape
(B=4, T=50, F=number-of-kept-sensors), and verify:

    outputs["weibull"]["k"].shape == (4, 50)
    outputs["weibull"]["lambda"].shape == (4, 50)
    no NaNs anywhere
    both k and lambda are strictly positive

Catch silent shape bugs HERE, before you spend an hour debugging the
training loop.
"""

# TODO: instantiate model
# TODO: random batch
# TODO: forward
# TODO: assertions on shapes, finiteness, positivity
raise NotImplementedError("Run the synthetic-batch sanity check")


---
# Part 4 — Training

## Weibull NLL with right-censoring

The negative log-likelihood for one observation $(t, u)$ given predicted
$(k, \lambda)$ is:

$$\mathcal{L}(t, u; k, \lambda) =
   -\,u \log f(t; k, \lambda) - (1 - u) \log S(t; k, \lambda)$$

For numerical stability, work directly on the log scale. Let
$z = (t / \lambda)^k$. Then:

$$\log S(t; k, \lambda) = -z$$
$$\log f(t; k, \lambda) = \log\!\frac{k}{\lambda} + (k-1) \log\!\frac{t}{\lambda} - z$$

Substituting:

$$\mathcal{L} = -\,u \left[\log k - \log \lambda + (k-1)(\log t - \log \lambda)\right] + z$$

This is well-defined for any $t > 0$, $k > 0$, $\lambda > 0$, and the
gradient is well-behaved.

<details>
<summary><b>How the censoring term emerges from the survival function</b></summary>

If we observe an event time $t$, the likelihood contribution is just the
PDF: $f(t)$.

If we only know the unit *survived past* $t$ (right-censored), the
likelihood contribution is the survival function $S(t) = P(T > t)$ — we
don't know exactly when (or even if) the event happens, only that it's
later than $t$.

Combining both cases with an indicator $u \in \{0, 1\}$:

$$\mathcal{L}_{\text{NLL}} = -\,u \log f(t) - (1-u) \log S(t)$$

This is the **Cox formulation** that underlies most modern survival
models, including Weibull AFT, DeepSurv, and WTTE-RNN.

</details>


In [ ]:
"""Numerically-stable Weibull negative log-likelihood with censoring.

Args:
    k:        (B, T) predicted shape, strictly positive
    lam:      (B, T) predicted scale, strictly positive
    t:        (B, T) target time-to-event, strictly positive
              (clip with eps to avoid log(0))
    censored: (B, T) float, 0 = OBSERVED (u=1), 1 = CENSORED (u=0)
    valid_mask: (B, T) bool, True at valid (non-padded) positions

Returns:
    scalar loss (mean over valid positions)

Implementation hints:
    1. eps = 1e-7. Clamp t to be at least eps.
    2. Compute log_t = log(t), log_lam = log(lam), log_k = log(k).
    3. z = (t / lam) ** k          # use exp(k * (log_t - log_lam)) for stability
    4. u = 1.0 - censored          # observed indicator
    5. log_pdf = log_k - log_lam + (k - 1) * (log_t - log_lam) - z
    6. log_surv = -z
    7. per_sample_nll = -(u * log_pdf + (1 - u) * log_surv)
    8. Mask to valid positions and average:
           loss = (per_sample_nll * valid_mask).sum() / valid_mask.sum().clamp(min=1)
"""


def weibull_nll_loss(
    k: torch.Tensor,
    lam: torch.Tensor,
    t: torch.Tensor,
    censored: torch.Tensor,
    valid_mask: torch.Tensor,
) -> torch.Tensor:
    # TODO
    raise NotImplementedError("Implement weibull_nll_loss")


## 4.3  Multi-head total loss

The total training objective is a weighted sum over registered heads:

$$\mathcal{L}_{\text{total}} = \sum_{i} w_i \cdot \mathcal{L}_i$$

```mermaid
flowchart LR
    H["model outputs<br/>dict[head_name → dict]"] --> W["Weibull head outputs<br/>(k, λ)"]
    H --> Y["YourHead outputs"]
    W --> LW["weibull_nll_loss<br/>× w_weibull"]
    Y --> LY["your_loss<br/>× w_yours"]
    LW --> SUM["Total loss"]
    LY --> SUM
```

In this notebook only the Weibull loss is active (the YourHeadHere stub
contributes nothing). If you implement an additional head later, register
its loss function here and give it a weight.


In [ ]:
"""Compute the weighted total loss across all registered heads.

Args:
    outputs: dict {head_name: head_output_dict} from TTETransformer
    batch:   dict from the DataLoader (targets, censored, key_padding_mask, …)
    head_weights: dict {head_name: float}

Returns:
    total: scalar loss
    components: dict {head_name: scalar loss} for logging

Implementation hints:
    1. valid_mask = ~batch["key_padding_mask"]      # True at REAL positions
    2. For "weibull" head: call weibull_nll_loss(...).
    3. For "your_head" head (if active and you've implemented its loss):
           your_head_loss(...)
    4. total = sum(w * components[name] for name, w in head_weights.items())
"""


def compute_total_loss(
    outputs: dict,
    batch: dict,
    head_weights: dict[str, float],
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    # TODO
    raise NotImplementedError("Aggregate weighted multi-head loss")


## 4.5  Optimizer, schedule, gradient clipping

Standard modern-transformer training recipe:

- **AdamW** — Adam with decoupled weight decay
- **Warmup + cosine annealing** — linearly warm up the learning rate
  over the first few hundred steps, then cosine-decay to a fraction of
  the peak
- **Gradient clipping** — clip the global norm to 1.0 to prevent
  occasional exploding-gradient spikes

These choices are not load-bearing for this notebook to *work*, but
they're what works in practice and make for cleaner loss curves.


In [ ]:
"""Optimizer and learning-rate schedule.

Implementation hints:
    - torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.01)
    - Use torch.optim.lr_scheduler.LambdaLR with a function that:
          step < warmup_steps:  return step / warmup_steps
          else:                 cosine decay from 1.0 to min_ratio
                                (e.g., 0.1) over remaining steps
"""

import math


def build_optimizer_and_scheduler(
    model,
    total_steps: int,
    peak_lr: float = 3e-4,
    warmup_ratio: float = 0.05,
    min_ratio: float = 0.1,
):
    # TODO: build optimizer
    # TODO: build LambdaLR scheduler with warmup + cosine
    raise NotImplementedError


In [ ]:
"""train_one_epoch and validate_one_epoch.

Implementation hints (train_one_epoch):
    1. Set the model to training mode: model.train()
    2. running = 0.0; n = 0
    3. for batch in train_loader:
           move batch tensors to DEVICE
           optimizer.zero_grad()
           outputs = model(batch["sensors"], key_padding_mask=batch["key_padding_mask"])
           loss, components = compute_total_loss(outputs, batch, head_weights)
           loss.backward()
           torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
           optimizer.step(); scheduler.step()
           running += loss.item() * batch_size; n += batch_size
    4. return running / n

Implementation hints (validate_one_epoch):
    Same shape but inside `with torch.no_grad():` and with
    `model.train(False)` (the inference-mode equivalent), no backward, no step.
"""


def train_one_epoch(model, loader, optimizer, scheduler, head_weights):
    # TODO
    raise NotImplementedError


def validate_one_epoch(model, loader, head_weights):
    # TODO
    raise NotImplementedError


In [ ]:
"""Main training loop.

Implementation hints:
    1. Instantiate model, move to DEVICE.
    2. Compute total_steps = num_epochs * len(train_loader).
    3. optimizer, scheduler = build_optimizer_and_scheduler(...)
    4. head_weights = {"weibull": 1.0}
    5. history = {"train": [], "val": []}
    6. for epoch in range(num_epochs):
           train_loss = train_one_epoch(...)
           val_loss = validate_one_epoch(...)
           history["train"].append(train_loss)
           history["val"].append(val_loss)
           print(f"epoch {epoch}: train={train_loss:.4f} val={val_loss:.4f}")

Suggested defaults for a first run:
    num_epochs = 20  (bump up later)
    peak_lr = 3e-4
"""

NUM_EPOCHS = 20

# TODO: training loop
raise NotImplementedError("Run the training loop")


In [ ]:
"""Plot train + validation loss curves.

A well-behaved Weibull NLL curve will:
    - drop steeply for the first few epochs
    - keep decreasing on train
    - flatten on validation (and ideally not increase — that would be overfitting)
"""

# TODO: plot
raise NotImplementedError("Plot loss curves")


---
# Part 5 — Evaluation

A survival model doesn't have a single "accuracy" number. We care about
three things:

1. **Discrimination** — does the model correctly RANK engines by risk?
   Measured by **concordance index (C-index)**: the probability that, for
   a randomly chosen pair where one engine failed before the other, the
   model assigned the failed engine a higher predicted hazard.
2. **Calibration** — when the model says "70% probability of failing
   within 50 cycles," does that actually happen 70% of the time? Measured
   by **Brier score** and calibration plots.
3. **Usefulness** — can we hand a maintenance engineer a number like
   "this engine will fail in 87 cycles (90% CI: 62–121)"?

We'll implement both metrics from scratch with numpy — they're short,
well-defined formulas, and writing them yourself is consistent with
building the rest of the architecture from scratch. Then we'll build the
distribution-to-number conversion machinery for (3) from scratch too.


In [ ]:
"""Concordance index (C-index) — from-scratch implementation, plus the
last-step prediction collector.

THE MATH:
    For every pair of engines (i, j) where true_time_i != true_time_j,
    the pair is CONCORDANT if the predicted ordering matches the true
    ordering (the engine that fails first is predicted to fail first).

        C-index = (concordant + 0.5 * tied_predictions) / comparable_pairs

    C = 0.5  → random ranking
    C = 1.0  → perfect ranking
    C = 0.0  → perfectly wrong (also useful — just invert the score)

NOTE on censoring: in the CMAPSS test setup we know the true RUL for
every test engine (from RUL_FD002.txt), so all pairs are comparable.
A fully-general C-index for right-censored data only counts pairs where
the SHORTER true time was OBSERVED (not censored). We're using the
simpler all-observed form here. If you later evaluate on truly censored
data, add that filter.

Implementation hints (concordance_index):
    1. i_idx, j_idx = numpy.triu_indices(len(true_times), k=1)
       → all pairs (i, j) with i < j
    2. true_diff = true_times[i_idx] - true_times[j_idx]
       pred_diff = predicted_times[i_idx] - predicted_times[j_idx]
    3. comparable = true_diff != 0
    4. concordant = (numpy.sign(true_diff) == numpy.sign(pred_diff)) & comparable
       tied_pred  = (pred_diff == 0) & comparable
    5. return (concordant.sum() + 0.5 * tied_pred.sum()) / comparable.sum()

Implementation hints (collect_last_step_predictions):
    1. Set the model to inference mode with model.train(False).
    2. Wrap the loop in `with torch.no_grad():`.
    3. For each batch, forward → grab the (k, lambda) at the LAST VALID
       (non-padded) position per engine using `lengths` from the batch dict.
    4. Pair each with the engine's true RUL (carried in the loader).
    5. Return as numpy arrays: k_last, lam_last, true_ruls.
"""


def concordance_index(true_times: numpy.ndarray,
                      predicted_times: numpy.ndarray) -> float:
    """C-index assuming all events are observed (CMAPSS test setup).

    Args:
        true_times: (N,) actual time-to-event (true RUL)
        predicted_times: (N,) predicted time-to-event (e.g., Weibull median).
            Higher = predicted to fail LATER. Both arrays use the same
            "longer = better survival" convention.

    Returns:
        C-index in [0, 1].
    """
    # TODO: build pair indices with numpy.triu_indices
    # TODO: compute true_diff and pred_diff
    # TODO: compute comparable / concordant / tied_pred masks
    # TODO: return (concordant + 0.5 * tied) / comparable
    raise NotImplementedError("Implement concordance_index from scratch")


def collect_last_step_predictions(model, loader):
    """Return per-engine (k_last, lam_last, true_rul) numpy arrays."""
    # TODO: model.train(False); torch.no_grad()
    # TODO: for each batch, forward, gather last-valid-position per engine
    # TODO: stack into numpy arrays of shape (n_engines,)
    raise NotImplementedError


# TODO: predictions = collect_last_step_predictions(model, test_loader)
# TODO: predicted_medians = weibull_median(k_last, lam_last)
# TODO: c = concordance_index(true_ruls, predicted_medians); print c


In [ ]:
"""Integrated Brier Score — from-scratch implementation.

THE MATH:
    At a fixed horizon t, the Brier score is the mean squared error
    between the BINARY event indicator and the predicted FAILURE
    probability:

        1_i(t) = 1 if engine i has failed by time t, else 0
        F_i(t) = 1 - S_i(t) = 1 - exp( -(t / lam_i)^k_i )

        BS(t) = (1/N) * sum_i ( 1_i(t) - F_i(t) )^2

    The Integrated Brier Score averages BS(t) over a grid of horizons:

        IBS = mean over t in t_grid of BS(t)

    Lower is better. 0 = perfect; 0.25 = the trivial "always 0.5" predictor.

Implementation hints:
    1. Build t_grid = numpy.arange(t_min, t_max, step), e.g., (10, 200, 5).
    2. Predicted survival, shape (N, len(t_grid)):
           ratio = t_grid[None, :] / lam[:, None]
           S = numpy.exp(-ratio ** k[:, None])
    3. Event indicator, shape (N, len(t_grid)):
           events = (true_ruls[:, None] <= t_grid[None, :]).astype(float)
    4. BS_per_t = ((events - (1 - S)) ** 2).mean(axis=0)
    5. IBS = BS_per_t.mean()
    6. Optionally plot BS_per_t against t_grid — interesting to see
       which horizons the model handles best.
"""


def brier_score_per_t(true_ruls: numpy.ndarray,
                      k: numpy.ndarray,
                      lam: numpy.ndarray,
                      t_grid: numpy.ndarray) -> numpy.ndarray:
    """Brier score at every t in t_grid; returns shape (len(t_grid),)."""
    # TODO: predicted survival matrix S of shape (N, len(t_grid))
    # TODO: event indicator matrix of shape (N, len(t_grid))
    # TODO: per-t Brier = mean over engines of (event - (1 - S))^2
    raise NotImplementedError


def integrated_brier_score(true_ruls: numpy.ndarray,
                           k: numpy.ndarray,
                           lam: numpy.ndarray,
                           t_grid: numpy.ndarray) -> float:
    # TODO: mean of brier_score_per_t(...) over t_grid
    raise NotImplementedError


# TODO: build t_grid (e.g., numpy.arange(10, 200, 5))
# TODO: bs_per_t = brier_score_per_t(true_ruls, k_last, lam_last, t_grid)
# TODO: ibs = integrated_brier_score(...); print ibs
# TODO: optionally pyplot.plot(t_grid, bs_per_t) to see BS as a function of horizon


## 5.4  From a distribution to a number with error bars

The Weibull quantile function (inverse CDF) gives the time $t$ at which
$P(T \le t) = p$:

$$Q(p; k, \lambda) = \lambda \cdot (-\ln(1 - p))^{1/k}$$

From this we derive everything we want:

| Quantity | Formula | Interpretation |
|---|---|---|
| Median | $\lambda (\ln 2)^{1/k}$ | 50% chance of failing earlier, 50% later |
| Mean | $\lambda \cdot \Gamma(1 + 1/k)$ | Expected cycles to failure |
| 5th percentile | $Q(0.05; k, \lambda)$ | Pessimistic lower bound |
| 95th percentile | $Q(0.95; k, \lambda)$ | Optimistic upper bound |
| 90% CI | $[Q(0.05), Q(0.95)]$ | Standard "error bar" |

The deliverable for an operator looks like:

> Engine 27 will fail in **87 cycles** (90% CI: 62–121).

```mermaid
flowchart LR
    K["predicted (k, λ)<br/>from Weibull head"] --> Q["Q(p; k, λ) =<br/>λ · (-ln(1-p))^(1/k)"]
    Q --> M["median: p=0.5"]
    Q --> L["lower: p=0.05"]
    Q --> U["upper: p=0.95"]
    M --> O["'87 cycles<br/>(90% CI: 62–121)'"]
    L --> O
    U --> O
```

The median is usually a better point estimate than the mean for survival
models — Weibull distributions are skewed, especially when $k$ is small.


In [ ]:
"""Quantile, median, and mean helpers for the Weibull distribution.

Implementation hints:
    quantile: lam * (-log(1 - p))^(1/k)
    median:   quantile(0.5)  →  lam * (ln 2)^(1/k)
    mean:     lam * Gamma(1 + 1/k)   (use scipy.special.gamma)

Operate on tensors (preserving shape and device).
"""

from scipy.special import gamma  # math.gamma also works for scalars


def weibull_quantile(k: torch.Tensor, lam: torch.Tensor, p: float) -> torch.Tensor:
    # TODO
    raise NotImplementedError


def weibull_median(k: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
    # TODO: quantile at 0.5
    raise NotImplementedError


def weibull_mean(k: torch.Tensor, lam: torch.Tensor) -> torch.Tensor:
    # TODO: lam * gamma(1 + 1/k)
    # Note: gamma is numpy-friendly; convert via .cpu().numpy() if needed,
    # OR use torch.exp(torch.lgamma(1 + 1/k)) which keeps it on-device.
    raise NotImplementedError


In [ ]:
"""Batch-convert per-engine predicted (k, λ) into
(point_estimate, lower_90, upper_90) numpy arrays.

Implementation hints:
    1. point = weibull_median(k, lam)
    2. lower = weibull_quantile(k, lam, 0.05)
    3. upper = weibull_quantile(k, lam, 0.95)
    4. Return as numpy arrays for plotting.

Run this on the test-set last-step predictions you collected earlier.
"""

# TODO: convert k_last, lam_last → (point, lower, upper) arrays
# TODO: print the first 10 predictions next to true RULs, e.g.:
#       "Engine  1: predicted 87 cycles  (90% CI: 62-121)  true RUL: 95"
raise NotImplementedError("Batch convert predictions to point + bounds")


## 5.7  Per-engine predicted PDF

A sanity check on what the model has actually learned: for a handful of
test engines, plot the Weibull PDF at the last observed cycle, mark the
median, shade the 90% CI, and show the true RUL.

A well-calibrated model:

- The true RUL falls inside the 90% CI most of the time
- The PDF is narrower for engines whose degradation pattern is
  unambiguous
- The median is close to the true RUL on average


In [ ]:
"""Plot the predicted Weibull PDF for ~6 sample test engines, marking
median, 90% CI, and true RUL.

Implementation hints:
    1. Pick 6 engines (e.g., indices 0, 1, 2, 50, 51, 100).
    2. For each, build t_grid = numpy.linspace(1, 4 * point_estimate, 200).
    3. PDF: f(t; k, lam) = (k/lam) * (t/lam)^(k-1) * exp(-(t/lam)^k)
    4. Plot in a 2x3 grid (subplots).
    5. axvline at median (solid), axvspan from lower_90 to upper_90 (shaded),
       axvline at true_RUL (dashed, different color).
    6. Title each subplot with engine index and k, lambda.
"""

# TODO: pick engines
# TODO: plot grid
raise NotImplementedError("Plot per-engine PDFs with median, CI, true RUL")


## 5.9  Forecast trajectory

This is the **payoff plot** of the whole notebook. Because the
transformer is causal and produces a prediction at every timestep, we
can ask:

*How does the predicted time-to-event — and its error bars — evolve as
the engine accumulates cycles?*

Two patterns you'd hope to see:

1. The **median forecast** stays near the actual remaining time (drifts
   downward by 1 cycle for every cycle that passes).
2. The **90% CI band tightens** as the engine approaches failure and the
   degradation signal becomes unambiguous.

This is the qualitative behavior that justifies the entire architectural
choice.


In [ ]:
"""For 2-3 sample test engines, plot how the predicted time-to-event and
its 90% CI evolve as cycles accumulate.

Implementation hints:
    1. Pick 2-3 engines from the test set.
    2. For each, run a forward pass on the engine's full observed sequence
       (use a batch of size 1, no padding mask).
    3. From the per-timestep (k_t, lam_t) outputs, compute:
           median_t = weibull_median(k_t, lam_t)
           lower_t  = weibull_quantile(k_t, lam_t, 0.05)
           upper_t  = weibull_quantile(k_t, lam_t, 0.95)
    4. Plot:
           x-axis: observed cycle (1, 2, …, T_observed)
           y-axis: predicted time-to-event
           line:   median over cycles
           shaded: lower → upper
           dashed line: TRUE remaining cycles at each t
                       (i.e., (T_observed + true_RUL) - t for each cycle t)
"""

# TODO: pick engines, plot trajectories
raise NotImplementedError("Plot forecast trajectories")


## 5.11  Calibration plot

A calibration plot bins predictions by their predicted survival
probability and checks the empirical survival rate within each bin.

A perfectly calibrated model gives a line $y = x$. Above the diagonal
means the model **underestimates** survival (predictions are too
pessimistic); below the diagonal means it **overestimates** (too
optimistic).


In [ ]:
"""Calibration plot: predicted vs empirical survival at a fixed horizon.

Implementation hints:
    1. Pick a horizon, e.g., t_horizon = 50 cycles.
    2. For each test engine, predicted_survival = S(t_horizon; k, lam).
    3. Empirical survival at t_horizon: did this engine's true RUL exceed
       50? (1 if true_rul > t_horizon, else 0).
    4. Bin predictions into ~10 quantile bins. In each bin, plot
       (mean predicted survival, mean empirical survival).
    5. Plot the y=x diagonal as a reference line.
"""

# TODO: calibration plot
raise NotImplementedError("Plot calibration curve")


In [ ]:
"""Headline results table.

For the test set, compute:
    - RMSE between predicted median and true RUL
    - NASA score (the asymmetric CMAPSS scoring function):
        let d = predicted - true
        score_i = exp(-d/13) - 1   if d < 0  (early prediction — better)
        score_i = exp( d/10) - 1   if d >= 0 (late prediction — worse)
        NASA score = sum of score_i over all test engines (lower is better)
    - C-index (computed earlier)
    - Integrated Brier Score (computed earlier)

Present as a pandas DataFrame for easy reading.
"""

# TODO: compute metrics
# TODO: assemble DataFrame and display
raise NotImplementedError("Compute and display headline metrics table")


---
# Part 6 — Extensions

You've built and trained a causal multivariate transformer with a
Weibull time-to-event head and proper survival-style evaluation. The
following extensions are deliberately left for you to explore.

## 6.1  Adding another dataset

The data loader contract is:

> Return a list of `(sensor_sequence: numpy.ndarray[T, F], targets: numpy.ndarray[T], censored_flags: numpy.ndarray[T])` tuples.

Any time-to-event dataset that can produce these tuples plugs into the
existing pipeline. To add FD001/FD003/FD004:

1. Change `SUBSET` in Part 2 and re-run.
2. Note that FD003 and FD004 have **two fault modes** — you may want to
   add a fault-mode classification head (good use case for
   `YourHeadHere`).
3. For a non-CMAPSS dataset, write a new `load_*` function that returns
   train/test DataFrames in the same column shape. Adjust the sensor
   selection threshold for the new sensor count.

## 6.2  Adding another head

The head registry is the seam. To add (say) a regime classifier:

1. Define a new `nn.Module` (subclass of `nn.Module`, takes
   `(B, T, d_model)`, returns a dict).
2. Define its loss function (e.g., `functional.cross_entropy` for classification).
3. Pass it into `TTETransformer` via the `extra_heads` argument:
   `extra_heads={"regime": RegimeClassifier(d_model, num_classes=6)}`.
4. Add a weight in `head_weights`, e.g.,
   `{"weibull": 1.0, "regime": 0.3}`.
5. Update `compute_total_loss` to call the new loss function on the new
   head's outputs.

The encoder, training loop, and most of the evaluation code don't
change.

## 6.3  Swapping causal for bidirectional

For OFFLINE analysis (e.g., post-hoc anomaly detection on completed
engine runs), you might want bidirectional attention. Two-line change:

1. In `MultiHeadAttention.forward`, accept a `causal` flag and skip the
   causal-mask construction when `causal=False`.
2. Pass `causal=False` down from `TransformerBlock.forward`.

For training, you'd also rethink the target — without causality, a
single "time to failure at end of sequence" target makes more sense
than per-timestep predictions.

## 6.4  Open exercises

- Visualize attention weights at different layers. Are early layers
  attending locally and later layers globally? Use the `attn` tensor
  that `scaled_dot_product_attention` already returns.
- Compare sinusoidal vs **learned** positional encoding
  (`nn.Embedding(max_len, d_model)`).
- Try a **deeper** vs **wider** model: 6 layers × $d_{\text{model}} = 64$
  vs 3 layers × $d_{\text{model}} = 128$. Which generalizes better on FD002?
- Re-run on FD004 and compare metrics. The increased complexity (two
  fault modes) should show up in a wider 90% CI.
- Add a **monotonicity penalty** on the predicted median —
  forecasts shouldn't increase as more degradation evidence arrives.

You've now built a working time-to-event transformer from scratch and
understand every component. The same architecture and training recipe
generalize to many other prognostics and reliability problems —
hard-drive failures, manufacturing yield, medical event prediction —
wherever you have multivariate time series and right-censored event
labels.

Happy hacking.
